# The Five-Minute Oscillations on the Solar Surface — Implementation / 구현

**Paper**: Ulrich, R. K. (1970). "The Five-Minute Oscillations on the Solar Surface." *The Astrophysical Journal*, 162, 993–1002.

This notebook implements the key physical concepts from Ulrich (1970):
이 노트북은 Ulrich (1970) 논문의 핵심 물리적 개념을 구현합니다:

1. **Solar model & characteristic frequencies / 태양 모델 및 특성 주파수**: Table 1 데이터 재현 및 시각화
2. **Wave trapping & reflecting layers / 파동 가둠 및 반사층**: Figure 1 재현 — 반사층 고도 vs. 수평 파장
3. **Dispersion relation & $k_h$-$\omega$ diagram / 분산 관계 및 진단 다이어그램**: Figure 2 재현 — 핵심 결과
4. **Energy balance & overstability / 에너지 균형 및 overstability**: 에너지 플럭스 분석

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Part 1: Solar Model & Characteristic Frequencies / 태양 모델 및 특성 주파수

Ulrich의 Table 1에서 태양 모델의 주요 물리량을 재현합니다. 이 모델은 광구 아래의 대류층 구조를 기술하며, 파동 가둠 분석의 기초가 됩니다.

We reproduce the key physical quantities from Ulrich's Table 1. This model describes the convection zone structure below the photosphere and forms the basis for wave trapping analysis.

주요 변수 / Key variables:
- $z$: 고도 (광학 깊이 1 기준) / Altitude (relative to optical depth unity)
- $T$: 온도 / Temperature (K)
- $c$: 음속 / Sound speed (km/s)
- $\omega_0$: 음향 차단 주파수 / Acoustic cutoff frequency (s$^{-1}$)
- $\omega_R$: 복사 상호작용률 / Radiative interaction rate (s$^{-1}$)
- $N^2$: Brunt-Väisälä 주파수의 제곱 / Square of Brunt-Väisälä frequency (s$^{-2}$)

In [ ]:
# Table 1 data from Ulrich (1970)
# Columns: -z (km), T (K), c (km/s), omega_0 * 10^2 (s^-1),
#           omega_R * 10^2 (s^-1), N^2 * 10^4 (s^-2)

table1_data = np.array([
    [239,   4640,  7.23,  3.23,  0.33,  10.1],
    [387,   4680,  7.23,  3.17,  1.10,   9.5],
    [474,   4820,  7.30,  3.00,  2.11,   8.7],
    [570,   5130,  7.56,  2.71,  3.57,   6.2],
    [685,   5940,  8.16,  2.40,  6.36,   4.5],
    [719,   6330,  8.41,  1.78,  9.14,   2.3],
    [750,   6830,  8.70,  1.13,  7.61,   3.9],
    [793,   7960,  9.12, -1.76,  4.47, -18.6],
    [821,   9280,  9.39, -3.69,  1.94, -30.4],
    [837,  10000,  9.61, -1.67,  0.72, -15.0],
    [863,  10700,  9.92,  0.08,  0.20,  -8.0],
    [908,  11300, 10.3,   0.77,  0.04,  -3.1],
    [1040, 12300, 10.9,   1.06,  0.00,  -1.0],
    [1220, 13200, 11.6,   1.10,  0.00,  -0.47],
    [1410, 14100, 12.2,   1.07,  0.00,  -0.27],
    [2730, 19300, 15.8,   0.87,  0.00,  -0.03],
    [4640, 27600, 20.9,   0.67,  0.00,   0.00],
    [7770, 45000, 29.6,   0.48,  0.00,   0.00],
    [17900, 119000, 50.8,  0.29, 0.00,   0.00],
    [49800, 755000, 95.2,  0.17, 0.00,   0.00],
])

# Extract columns (note: z is negative, going below photosphere)
z_km = -table1_data[:, 0]          # altitude in km (negative = below surface)
T_K = table1_data[:, 1]            # temperature in K
c_kms = table1_data[:, 2]          # sound speed in km/s
omega_0 = table1_data[:, 3] * 1e-2 # acoustic cutoff frequency in s^-1
omega_R = table1_data[:, 4] * 1e-2 # radiative interaction rate in s^-1
N2 = table1_data[:, 5] * 1e-4      # Brunt-Vaisala frequency squared in s^-2

print("Table 1: Important Properties of the Average Model")
print("=" * 80)
print(f"{'z (km)':>10} {'T (K)':>8} {'c (km/s)':>10} {'ω₀ (s⁻¹)':>12} "
      f"{'ωR (s⁻¹)':>12} {'N² (s⁻²)':>12}")
print("-" * 80)
for i in range(len(z_km)):
    print(f"{z_km[i]:>10.0f} {T_K[i]:>8.0f} {c_kms[i]:>10.2f} "
          f"{omega_0[i]:>12.4f} {omega_R[i]:>12.4f} {N2[i]:>12.6f}")

In [ ]:
# Visualize characteristic frequencies as a function of depth
# 깊이에 따른 특성 주파수 시각화

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Temperature and sound speed profiles
ax1 = axes[0, 0]
ax1_twin = ax1.twinx()
ln1 = ax1.plot(-z_km, T_K, 'r-o', ms=4, label='T (K)')
ln2 = ax1_twin.plot(-z_km, c_kms, 'b-s', ms=4, label='c (km/s)')
ax1.set_xlabel('Depth below photosphere / 광구 아래 깊이 (km)')
ax1.set_ylabel('Temperature / 온도 (K)', color='r')
ax1_twin.set_ylabel('Sound speed / 음속 (km/s)', color='b')
ax1.set_xscale('log')
ax1.set_title('(a) Temperature & Sound Speed / 온도 및 음속')
lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

# (b) Acoustic cutoff frequency
ax2 = axes[0, 1]
ax2.plot(-z_km, omega_0, 'g-o', ms=4)
ax2.axhline(y=2*np.pi/300, color='r', ls='--', alpha=0.7,
            label=f'ω(5 min) = {2*np.pi/300:.4f} s⁻¹')
ax2.axhline(y=2*np.pi/200, color='orange', ls='--', alpha=0.7,
            label=f'ω(200 s) = {2*np.pi/200:.4f} s⁻¹')
ax2.set_xlabel('Depth below photosphere / 광구 아래 깊이 (km)')
ax2.set_ylabel('$\\omega_0$ (s$^{-1}$)')
ax2.set_xscale('log')
ax2.set_title('(b) Acoustic Cutoff Frequency / 음향 차단 주파수 $\\omega_0$')
ax2.legend(fontsize=9)

# (c) Radiative interaction rate
ax3 = axes[1, 0]
ax3.plot(-z_km, omega_R, 'm-o', ms=4)
ax3.set_xlabel('Depth below photosphere / 광구 아래 깊이 (km)')
ax3.set_ylabel('$\\omega_R$ (s$^{-1}$)')
ax3.set_xscale('log')
ax3.set_title('(c) Radiative Interaction Rate / 복사 상호작용률 $\\omega_R$')

# (d) Brunt-Vaisala frequency squared
ax4 = axes[1, 1]
ax4.plot(-z_km, N2, 'k-o', ms=4)
ax4.axhline(y=0, color='gray', ls='-', alpha=0.5)
ax4.fill_between(-z_km, N2, 0, where=np.array(N2) > 0,
                 alpha=0.2, color='blue', label='Stable (N² > 0)')
ax4.fill_between(-z_km, N2, 0, where=np.array(N2) < 0,
                 alpha=0.2, color='red', label='Convectively unstable (N² < 0)')
ax4.set_xlabel('Depth below photosphere / 광구 아래 깊이 (km)')
ax4.set_ylabel('$N^2$ (s$^{-2}$)')
ax4.set_xscale('log')
ax4.set_title('(d) Brunt-Väisälä Frequency² / Brunt-Väisälä 주파수²')
ax4.legend(fontsize=9)

plt.suptitle('Solar Model Properties from Ulrich (1970) Table 1\n'
             'Ulrich (1970) Table 1의 태양 모델 물리량', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 2: Wave Trapping — Dispersion Relation / 파동 가둠 — 분산 관계

Whitaker (1963)의 국소 분산 관계를 사용하여 파동이 전파할 수 있는 영역과 반사되는 영역을 계산합니다.

Using Whitaker's (1963) local dispersion relation, we compute the regions where waves can propagate and where they are reflected.

$$k_z^2 = \frac{\omega^2 - \omega_0^2}{c^2} - k_h^2\left(1 - \frac{N^2}{\omega^2}\right) \tag{1}$$

파동이 전파하려면 $k_z^2 > 0$, 반사(evanescent)되려면 $k_z^2 < 0$이어야 합니다.
$k_z^2 = 0$인 경계가 바로 **반사층(reflecting layer)**입니다.

For wave propagation $k_z^2 > 0$, for reflection (evanescent) $k_z^2 < 0$.
The boundary where $k_z^2 = 0$ is the **reflecting layer**.

In [ ]:
# Build interpolators for the solar model
# 태양 모델에 대한 보간 함수 구축

# Use depth (positive, in km) as the independent variable
depth_km = -z_km  # positive depth below photosphere

# Interpolate on log(depth) for smooth profiles
log_depth = np.log10(depth_km)

c_interp = interp1d(log_depth, c_kms, kind='cubic', fill_value='extrapolate')
omega0_interp = interp1d(log_depth, omega_0, kind='cubic', fill_value='extrapolate')
N2_interp = interp1d(log_depth, N2, kind='cubic', fill_value='extrapolate')

# Fine grid for plotting
depth_fine = np.logspace(np.log10(239), np.log10(49800), 500)
log_depth_fine = np.log10(depth_fine)

c_fine = c_interp(log_depth_fine)
omega0_fine = omega0_interp(log_depth_fine)
N2_fine = N2_interp(log_depth_fine)


def kz_squared(omega, kh, depth_val):
    """Compute k_z^2 from the local dispersion relation (eq. 1).

    Args:
        omega: Angular frequency (s^-1).
        kh: Horizontal wavenumber (km^-1).
        depth_val: Depth below photosphere (km).

    Returns:
        k_z^2 in km^-2.
    """
    ld = np.log10(depth_val)
    c = c_interp(ld)
    w0 = omega0_interp(ld)
    n2 = N2_interp(ld)

    term1 = (omega**2 - w0**2) / c**2
    term2 = kh**2 * (1 - n2 / omega**2)
    return term1 - term2


# Demonstrate: k_z^2 as a function of depth for a 5-minute oscillation
# 5분 진동에 대한 k_z^2의 깊이 의존성 시연
omega_5min = 2 * np.pi / 300  # 5-minute oscillation angular frequency

fig, ax = plt.subplots(figsize=(10, 6))

for kh_val, color, label in [(0.0005, 'blue', '$k_h = 5 \\times 10^{-4}$ km$^{-1}$'),
                               (0.001, 'green', '$k_h = 10^{-3}$ km$^{-1}$'),
                               (0.002, 'red', '$k_h = 2 \\times 10^{-3}$ km$^{-1}$')]:
    kz2_vals = [kz_squared(omega_5min, kh_val, d) for d in depth_fine]
    ax.plot(depth_fine, kz2_vals, color=color, label=label, lw=2)

ax.axhline(y=0, color='black', ls='-', lw=1)
ax.fill_between(depth_fine, -1e-5, 0, alpha=0.1, color='gray')
ax.set_xscale('log')
ax.set_xlabel('Depth below photosphere / 광구 아래 깊이 (km)')
ax.set_ylabel('$k_z^2$ (km$^{-2}$)')
ax.set_title('$k_z^2$ vs Depth for 5-Minute Oscillation ($\\omega = 2\\pi/300$ s$^{-1}$)\n'
             '5분 진동에 대한 $k_z^2$의 깊이 의존성')
ax.set_ylim(-2e-5, 3e-5)
ax.legend()
ax.text(300, -1.5e-5, 'Evanescent\n(반사 영역)', fontsize=11, ha='center',
        color='gray', style='italic')
ax.text(3000, 2e-5, 'Propagating\n(전파 영역)', fontsize=11, ha='center',
        color='black', style='italic')
plt.tight_layout()
plt.show()

### Reflecting Layer Altitude (Figure 1 재현)

반사층의 고도를 수평 파장의 함수로 그립니다. $k_z^2 = 0$을 만족하는 깊이를 각 $(\omega, k_h)$ 조합에 대해 수치적으로 구합니다.

We plot the reflecting layer altitude as a function of horizontal wavelength. For each $(\omega, k_h)$ combination, we numerically find the depth where $k_z^2 = 0$.

**상부 반사면**: 광구 근처에서 $\omega_0$가 $\omega$보다 클 때 형성
**하부 반전점**: 깊은 곳에서 음속 증가로 인한 굴절에 의해 형성

**Upper reflecting surface**: formed near photosphere when $\omega_0 > \omega$
**Lower turning point**: formed at depth due to refraction from increasing sound speed

In [ ]:
def find_reflecting_layers(omega, kh, depth_range=(240, 49000), n_points=1000):
    """Find depths where k_z^2 = 0 (reflecting layers).

    Args:
        omega: Angular frequency (s^-1).
        kh: Horizontal wavenumber (km^-1).
        depth_range: (min_depth, max_depth) in km.
        n_points: Number of grid points for root search.

    Returns:
        List of depths (km) where k_z^2 = 0.
    """
    depths = np.logspace(np.log10(depth_range[0]), np.log10(depth_range[1]),
                         n_points)
    kz2_vals = np.array([kz_squared(omega, kh, d) for d in depths])

    roots = []
    for i in range(len(kz2_vals) - 1):
        if kz2_vals[i] * kz2_vals[i + 1] < 0:
            try:
                root = brentq(lambda d: kz_squared(omega, kh, d),
                              depths[i], depths[i + 1])
                roots.append(root)
            except ValueError:
                pass
    return roots


# Reproduce Figure 1: reflecting layer altitude vs horizontal wavelength
# Figure 1 재현: 반사층 고도 vs 수평 파장

periods = [180, 200, 300]  # oscillation periods in seconds
colors = ['blue', 'green', 'red']
labels = ['180 s', '200 s', '300 s (5 min)']

fig, ax = plt.subplots(figsize=(10, 8))

lambda_h_range = np.logspace(np.log10(1000), np.log10(30000), 100)

for period, color, label in zip(periods, colors, labels):
    omega = 2 * np.pi / period
    upper_layers = []
    lower_layers = []
    upper_lambdas = []
    lower_lambdas = []

    for lam_h in lambda_h_range:
        kh = 2 * np.pi / lam_h  # convert wavelength to wavenumber
        roots = find_reflecting_layers(omega, kh)

        if len(roots) >= 1:
            # Upper reflecting layer (shallowest)
            upper_layers.append(roots[0])
            upper_lambdas.append(lam_h)
        if len(roots) >= 2:
            # Lower turning point (deepest)
            lower_layers.append(roots[-1])
            lower_lambdas.append(lam_h)

    if upper_lambdas:
        ax.plot(np.array(upper_lambdas) / 1e3,
                -np.array(upper_layers) / 1e3,
                color=color, lw=2, label=f'{label} (upper / 상부)')
    if lower_lambdas:
        ax.plot(np.array(lower_lambdas) / 1e3,
                -np.array(lower_layers) / 1e3,
                color=color, lw=2, ls='--', label=f'{label} (lower / 하부)')

ax.set_xlabel('Horizontal wavelength $\\lambda_h$ / 수평 파장 ($10^3$ km)')
ax.set_ylabel('Altitude $z$ / 고도 ($10^3$ km below photosphere)')
ax.set_title('Reflecting Layer Altitude vs Horizontal Wavelength (cf. Ulrich Fig. 1)\n'
             '반사층 고도 vs 수평 파장 (Ulrich Fig. 1 비교)')
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Part 3: $k_h$-$\omega$ Diagnostic Diagram (Figure 2 재현) / 진단 다이어그램

이것이 Ulrich 논문의 **핵심 결과**입니다. 공진 캐비티에서 정상파 조건을 만족하는 $(\omega, k_h)$ 조합을 찾습니다.

This is the **key result** of Ulrich's paper. We find the $(\omega, k_h)$ combinations that satisfy the standing wave condition in the resonant cavity.

공진 조건은 상부와 하부 반사면 사이에서 반경 방향 파수의 적분이 반정수 배의 $\pi$가 되는 것입니다:

The resonance condition requires that the integral of the radial wavenumber between the upper and lower reflecting surfaces equals a half-integer multiple of $\pi$:

$$\int_{r_1}^{r_2} k_z \, dr = \left(n + \frac{1}{2}\right)\pi$$

여기서 $n = 0, 1, 2, \ldots$은 radial order (modal number)입니다.

where $n = 0, 1, 2, \ldots$ is the radial order (modal number).

In [ ]:
def phase_integral(omega, kh, depth_range=(240, 49000), n_points=2000):
    """Compute the phase integral of k_z between reflecting layers.

    The phase integral determines modal number:
    integral(k_z dz) = (n + 1/2) * pi

    Args:
        omega: Angular frequency (s^-1).
        kh: Horizontal wavenumber (km^-1).
        depth_range: Search range for reflecting layers.
        n_points: Integration grid resolution.

    Returns:
        Phase integral value, or None if no cavity exists.
    """
    roots = find_reflecting_layers(omega, kh, depth_range, n_points=500)

    if len(roots) < 2:
        return None

    # Integrate k_z between the innermost pair of reflecting layers
    z1 = roots[0]
    z2 = roots[-1]

    depths = np.linspace(z1, z2, n_points)
    kz2_vals = np.array([kz_squared(omega, kh, d) for d in depths])

    # Only integrate where k_z^2 > 0 (propagating region)
    kz_vals = np.where(kz2_vals > 0, np.sqrt(kz2_vals), 0.0)

    integral = np.trapz(kz_vals, depths)
    return integral


def find_eigenfrequency(kh, n_mode, omega_range=(0.012, 0.035),
                        n_search=200):
    """Find eigenfrequency for given kh and modal number n.

    Args:
        kh: Horizontal wavenumber (km^-1).
        n_mode: Modal number (0, 1, 2, ...).
        omega_range: Search range for omega (s^-1).
        n_search: Number of search points.

    Returns:
        Eigenfrequency omega (s^-1), or None.
    """
    target = (n_mode + 0.5) * np.pi

    omegas = np.linspace(omega_range[0], omega_range[1], n_search)
    phases = []
    valid_omegas = []

    for omega in omegas:
        phi = phase_integral(omega, kh)
        if phi is not None:
            phases.append(phi)
            valid_omegas.append(omega)

    if len(phases) < 2:
        return None

    phases = np.array(phases)
    valid_omegas = np.array(valid_omegas)

    # Find where phase integral crosses the target value
    for i in range(len(phases) - 1):
        if (phases[i] - target) * (phases[i + 1] - target) < 0:
            try:
                omega_root = brentq(
                    lambda w: phase_integral(w, kh) - target,
                    valid_omegas[i], valid_omegas[i + 1],
                    xtol=1e-6
                )
                return omega_root
            except (ValueError, TypeError):
                pass
    return None


# Compute eigenfrequencies for modal numbers 1-4
# 모달 번호 1-4에 대한 고유 주파수 계산
print("Computing k_h-ω diagram ridges... (this may take a minute)")
print("k_h-ω 다이어그램 능선 계산 중... (약 1분 소요)")

kh_values = np.linspace(0.0003, 0.003, 40)
modal_numbers = [1, 2, 3, 4]
ridge_data = {n: {'kh': [], 'omega': []} for n in modal_numbers}

for kh in kh_values:
    for n in modal_numbers:
        omega_eigen = find_eigenfrequency(kh, n)
        if omega_eigen is not None:
            ridge_data[n]['kh'].append(kh)
            ridge_data[n]['omega'].append(omega_eigen)

print("Done! / 완료!")

In [ ]:
# Plot the k_h-omega diagnostic diagram (Figure 2)
# k_h-omega 진단 다이어그램 (Figure 2) 그리기

fig, ax = plt.subplots(figsize=(10, 8))

colors_ridge = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for n, color in zip(modal_numbers, colors_ridge):
    kh_arr = np.array(ridge_data[n]['kh'])
    omega_arr = np.array(ridge_data[n]['omega'])
    if len(kh_arr) > 0:
        ax.plot(kh_arr, omega_arr, '-o', color=color, ms=4, lw=2,
                label=f'Mode {n}')

# Mark the 5-minute oscillation frequency range
ax.axhspan(2*np.pi/350, 2*np.pi/250, alpha=0.1, color='yellow',
           label='5-min oscillation range\n5분 진동 범위')
ax.axhline(y=2*np.pi/300, color='gray', ls=':', alpha=0.5)

# Reference: acoustic cutoff at photosphere
omega0_max = max(omega_0)
ax.axhline(y=omega0_max, color='red', ls='--', alpha=0.5,
           label=f'$\\omega_0$ max (photosphere) = {omega0_max:.3f} s⁻¹')

ax.set_xlabel('Horizontal wavenumber $k_h$ / 수평 파수 (km$^{-1}$)')
ax.set_ylabel('Angular frequency $\\omega$ / 각주파수 (s$^{-1}$)')
ax.set_title('Diagnostic $(k_h, \\omega)$ Diagram — Ulrich (1970) Figure 2\n'
             '진단 $(k_h, \\omega)$ 다이어그램 — Ulrich (1970) Figure 2')
ax.legend(loc='upper left', fontsize=9)

# Add period scale on right axis
ax2 = ax.twinx()
omega_ticks = ax.get_yticks()
omega_ticks = omega_ticks[omega_ticks > 0]
period_ticks = 2 * np.pi / omega_ticks
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(omega_ticks)
ax2.set_yticklabels([f'{p:.0f}' for p in period_ticks])
ax2.set_ylabel('Period / 주기 (s)')

# Add wavelength scale on top axis
ax3 = ax.twiny()
kh_ticks = np.array([0.0005, 0.001, 0.002, 0.003])
lambda_ticks = 2 * np.pi / kh_ticks / 1e3  # in 10^3 km
ax3.set_xlim(ax.get_xlim())
ax3.set_xticks(kh_ticks)
ax3.set_xticklabels([f'{l:.1f}' for l in lambda_ticks])
ax3.set_xlabel('Horizontal wavelength $\\lambda_h$ / 수평 파장 ($10^3$ km)')

plt.tight_layout()
plt.show()

# Print eigenfrequencies as periods
print("\nEigenfrequencies (as periods) / 고유 주파수 (주기로 표시):")
print("=" * 60)
for n in modal_numbers:
    if ridge_data[n]['omega']:
        periods_s = [2*np.pi/w for w in ridge_data[n]['omega']]
        print(f"  Mode {n}: {min(periods_s):.0f} – {max(periods_s):.0f} s "
              f"({min(periods_s)/60:.1f} – {max(periods_s)/60:.1f} min)")

## Part 4: Penetration Depth & Helioseismology Principle / 침투 깊이와 일진학 원리

서로 다른 spherical harmonic degree $\ell$ (또는 $k_h$)의 모드가 태양의 서로 다른 깊이를 탐사한다는 것이 helioseismology의 핵심 원리입니다. 하부 반전점의 깊이는:

The fundamental principle of helioseismology is that modes with different spherical harmonic degree $\ell$ (or $k_h$) probe different depths of the Sun. The depth of the lower turning point is:

$$\frac{c_s(r_t)}{r_t} = \frac{\omega}{\sqrt{\ell(\ell+1)}}$$

낮은 $\ell$ → 작은 $k_h$ → 더 깊은 침투 / Low $\ell$ → small $k_h$ → deeper penetration

In [ ]:
# Compute lower turning point depth as a function of k_h
# k_h의 함수로서 하부 반전점 깊이 계산

R_sun = 6.96e5  # solar radius in km
omega_5min = 2 * np.pi / 300

# For the simple approximation: lower turning point where c/r = omega/sqrt(l(l+1))
# Using k_h = sqrt(l(l+1)) / R_sun, so c(r_t)/r_t = omega * R_sun * k_h / sqrt(l(l+1))
# Simplified: we find where k_z^2 = 0 at depth

kh_range = np.linspace(0.0003, 0.005, 50)
lower_tp_depths = []
valid_kh = []

for kh in kh_range:
    roots = find_reflecting_layers(omega_5min, kh)
    if len(roots) >= 2:
        lower_tp_depths.append(roots[-1])
        valid_kh.append(kh)

# Convert k_h to approximate spherical harmonic degree l
# k_h ≈ sqrt(l(l+1)) / R_sun ≈ l / R_sun for large l
ell_approx = np.array(valid_kh) * R_sun

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(valid_kh, np.array(lower_tp_depths) / 1e3, 'b-o', ms=4, lw=2)
ax1.set_xlabel('$k_h$ (km$^{-1}$)')
ax1.set_ylabel('Lower turning point depth / 하부 반전점 깊이 ($10^3$ km)')
ax1.set_title('Penetration Depth vs Horizontal Wavenumber '
              '($\\omega = 2\\pi/300$ s$^{-1}$)\n'
              '침투 깊이 vs 수평 파수 (5분 진동)')

# Add approximate l scale on top
ax2 = ax1.twiny()
ax2.set_xlim(ax1.get_xlim())
l_ticks_pos = np.array([0.0005, 0.001, 0.002, 0.003, 0.004])
l_ticks_val = l_ticks_pos * R_sun
ax2.set_xticks(l_ticks_pos)
ax2.set_xticklabels([f'{l:.0f}' for l in l_ticks_val])
ax2.set_xlabel('Approximate $\\ell$ / 근사 구면 조화 차수')

# Annotate key depths
if lower_tp_depths:
    ax1.axhline(y=200, color='orange', ls='--', alpha=0.5,
                label='Convection zone base ~200,000 km\n대류층 바닥 ~200,000 km')

ax1.legend(fontsize=9)
ax1.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nHelioseismology principle / 일진학 원리:")
print("=" * 60)
print("Small k_h (low ℓ) → deep penetration → probes solar core")
print("작은 k_h (낮은 ℓ) → 깊은 침투 → 태양 핵 탐사")
print("Large k_h (high ℓ) → shallow penetration → probes near surface")
print("큰 k_h (높은 ℓ) → 얕은 침투 → 표면 근처 탐사")

## Part 5: Energy Balance — Overstability Analysis / 에너지 균형 — Overstability 분석

Ulrich의 Table 2를 재현합니다. 관측된 속도 진폭 $v_{\rm rms} = 0.2$ km/s를 각 모드에 할당하고, 에너지 소산율을 수평 파장별로 계산합니다.

We reproduce Ulrich's Table 2. We assign the observed rms velocity amplitude of $v_{\rm rms} = 0.2$ km/s to each mode and compute the energy dissipation rate as a function of horizontal wavelength.

에너지 플럭스(단위 면적당) / Energy flux (per unit area):
$$F \sim \rho \cdot v_{\rm rms}^2 \cdot c_g$$

여기서 $c_g$는 군속도(group velocity)의 수직 성분입니다.

where $c_g$ is the vertical component of the group velocity.

In [ ]:
# Reproduce Table 2: Required dissipation as a function of horizontal wavelength
# Table 2 재현: 수평 파장의 함수로서 필요 소산

# Table 2 data from the paper
# lambda_h (km), modal number, flux (10^6 ergs/cm^2/s), period (s)
table2_data = {
    4.83e3: {  # lambda_h = 4.83 x 10^3 km
        1: {'flux': -0.3, 'period': 311},
        2: {'flux': 7.4,  'period': 256},
        3: {'flux': 7.4,  'period': 211},
    },
    6.98e3: {  # lambda_h = 6.98 x 10^3 km
        1: {'flux': 0.0,  'period': 399},
        2: {'flux': 6.5,  'period': 313},
        3: {'flux': 6.3,  'period': 256},
        4: {'flux': 5.0,  'period': 219},
    },
    12.57e3: {  # lambda_h = 12.57 x 10^3 km
        1: {'flux': -3.5, 'period': 532},
        2: {'flux': -2.0, 'period': 412},
        3: {'flux': 1.2,  'period': 335},
        4: {'flux': 3.7,  'period': 282},
    },
}

# Visualize Table 2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# (a) Energy flux by modal number for each wavelength
wavelengths = sorted(table2_data.keys())
colors_wl = ['#1f77b4', '#ff7f0e', '#2ca02c']
markers = ['o', 's', '^']

for wl, color, marker in zip(wavelengths, colors_wl, markers):
    modes = sorted(table2_data[wl].keys())
    fluxes = [table2_data[wl][m]['flux'] for m in modes]
    ax1.plot(modes, fluxes, f'-{marker}', color=color, ms=8, lw=2,
             label=f'$\\lambda_h$ = {wl/1e3:.2f} × 10³ km')

ax1.axhline(y=0, color='gray', ls='-', lw=0.5)
ax1.axhline(y=5.6, color='red', ls='--', alpha=0.7,
            label='Athay (1966) observed loss\n'
                  '= 5.6 × 10⁶ ergs cm⁻² s⁻¹')
ax1.set_xlabel('Modal number / 모달 번호')
ax1.set_ylabel('Energy flux / 에너지 플럭스 (10⁶ ergs cm⁻² s⁻¹)')
ax1.set_title('Energy Flux by Mode & Wavelength\n'
              '모드 및 파장별 에너지 플럭스')
ax1.legend(fontsize=8)
ax1.set_xticks([1, 2, 3, 4])

# (b) Period by modal number
for wl, color, marker in zip(wavelengths, colors_wl, markers):
    modes = sorted(table2_data[wl].keys())
    periods_s = [table2_data[wl][m]['period'] for m in modes]
    ax2.plot(modes, periods_s, f'-{marker}', color=color, ms=8, lw=2,
             label=f'$\\lambda_h$ = {wl/1e3:.2f} × 10³ km')

ax2.axhspan(250, 350, alpha=0.15, color='yellow',
            label='5-min range (250–350 s)')
ax2.set_xlabel('Modal number / 모달 번호')
ax2.set_ylabel('Period / 주기 (s)')
ax2.set_title('Oscillation Period by Mode & Wavelength\n'
              '모드 및 파장별 진동 주기')
ax2.legend(fontsize=8)
ax2.set_xticks([1, 2, 3, 4])

plt.suptitle('Ulrich (1970) Table 2: Energy Dissipation Analysis\n'
             'Ulrich (1970) Table 2: 에너지 소산 분석', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey finding / 핵심 발견:")
print("=" * 60)
print("Modes with periods ≈ 300 s and intermediate wavelengths")
print("produce the most energy flux (~6-7 × 10⁶ ergs cm⁻² s⁻¹),")
print("matching the observed chromospheric energy loss rate of")
print("5.6 × 10⁶ ergs cm⁻² s⁻¹ (Athay 1966).")
print()
print("주기 ≈ 300초이고 중간 파장을 가진 모드가 가장 많은")
print("에너지 플럭스(~6-7 × 10⁶ ergs cm⁻² s⁻¹)를 생산하며,")
print("이는 관측된 채색층 에너지 손실률 5.6 × 10⁶ ergs cm⁻² s⁻¹")
print("(Athay 1966)과 일치합니다.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Dispersion relation / 분산 관계 | Whitaker (1963) local form, eq. (1) | Full linearized oscillation equations in spherical geometry (Unno et al. 1989) |
| Solar model / 태양 모델 | 20-point tabulated model (Table 1) | Standard Solar Model (Model S, Christensen-Dalsgaard et al. 1996) with >2000 mesh points |
| Eigenfrequency computation / 고유 주파수 계산 | Phase integral + numerical shooting | ADIPLS, GYRE — full eigenvalue solvers for stellar oscillations |
| $k_h$-$\omega$ diagram / 진단 다이어그램 | Predicted discrete ridges for modes 1–4 | Millions of observed modes from SDO/HMI, SOHO/MDI, GONG |
| Energy balance / 에너지 균형 | Analytical flux comparison showing overstability | Stochastic excitation by turbulent convection (Goldreich & Keeley 1977) is now the accepted mechanism |
| Wave trapping / 파동 가둠 | Upper reflection by $\omega_0$, lower by sound speed gradient | Same physics, refined with better solar models |

### 현대적 관점에서의 수정 사항 / Modern corrections

1. **여기 메커니즘 / Excitation mechanism**: Ulrich의 overstability (자체 여기) 해석은 현재는 수정되었습니다. 현대적 이해에서 p-mode는 대류에 의한 **확률적 여기(stochastic excitation)**로 구동됩니다 (Goldreich & Keeley 1977).
   Ulrich's overstability (self-excitation) interpretation has been revised. In modern understanding, p-modes are driven by **stochastic excitation** from convection.

2. **코로나 가열 / Coronal heating**: 5분 진동만으로는 코로나 가열을 설명하기 어렵습니다. 현재는 자기장을 통한 에너지 전달(nanoflares, Alfvén waves)이 주요 메커니즘으로 여겨집니다.
   5-minute oscillations alone cannot explain coronal heating. Currently, energy transfer through magnetic fields (nanoflares, Alfvén waves) is considered the primary mechanism.

3. **$k$-$\omega$ diagram의 정밀도 / Precision of $k$-$\omega$ diagram**: 현대 관측은 수백만 개의 모드를 식별하여 태양 내부의 음속, 밀도, 회전 프로파일을 0.1% 이내의 정밀도로 결정합니다.
   Modern observations identify millions of modes, determining the Sun's internal sound speed, density, and rotation profiles to within 0.1% precision.